# Part 2: Gnostic Distribution Functions (GDF)

This notebook teaches ELDF/EGDF using Anscombe data and reproduces the presentation plot:
- top row: `varS=False`
- bottom row: `varS=True`

Final output file:
- `eldf_vs_cdf_pdf_y_2x4.png`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

from machinegnostics.data import make_anscombe_check_data
from machinegnostics.magcal import ELDF, EGDF

plt.style.use("seaborn-v0_8")

## Step 1: Load y-values from all datasets

In [ ]:
datasets_y = {}
for ds_id in [1, 2, 3, 4]:
    _, y = make_anscombe_check_data(ds_id)
    datasets_y[ds_id] = np.asarray(y, dtype=float)

print({k: v.shape for k, v in datasets_y.items()})

## Step 2: Understand ELDF and EGDF on one dataset

In [ ]:
y_demo = datasets_y[1]

egdf_demo = EGDF(flush=False)
egdf_demo.fit(y_demo)

eldf_demo = ELDF(flush=False)
eldf_demo.fit(y_demo)

print("EGDF and ELDF fitted on Dataset 1 (y).")

## Step 3: Helper to create one subplot
We overlay:
- ELDF CDF (blue)
- empirical CDF (green)
- ELDF PDF (light orange, secondary y-axis)
- KDE PDF (red, secondary y-axis)

In [ ]:
def fit_eldf_with_vars(y, varS_flag):
    y = np.asarray(y, dtype=float)

    # Compatibility path: support multiple ELDF signatures across package versions.
    try:
        eldf = ELDF(flush=False, varS=varS_flag)
        eldf.fit(y)
        return eldf
    except TypeError:
        eldf = ELDF(flush=False)
        try:
            eldf.fit(y, varS=varS_flag)
        except TypeError:
            eldf.fit(y)
        return eldf


def plot_eldf_panel(ax, y, ds_id, varS_flag):
    y = np.asarray(y, dtype=float)
    y_sorted = np.sort(y)
    n = len(y_sorted)
    empirical_cdf = np.arange(1, n + 1) / n

    eldf = fit_eldf_with_vars(y, varS_flag)

    x_curve = None
    cdf_curve = None
    pdf_curve = None

    if hasattr(eldf, "di_points") and getattr(eldf, "di_points") is not None:
        x_curve = np.asarray(eldf.di_points, dtype=float).reshape(-1)
    elif hasattr(eldf, "data") and getattr(eldf, "data") is not None:
        x_curve = np.asarray(eldf.data, dtype=float).reshape(-1)

    if hasattr(eldf, "eldf_points") and getattr(eldf, "eldf_points") is not None:
        cdf_curve = np.asarray(eldf.eldf_points, dtype=float).reshape(-1)
    elif isinstance(getattr(eldf, "params", None), dict):
        cdf_curve = np.asarray(eldf.params.get("eldf"), dtype=float).reshape(-1)

    if hasattr(eldf, "pdf_points") and getattr(eldf, "pdf_points") is not None:
        pdf_curve = np.asarray(eldf.pdf_points, dtype=float).reshape(-1)
    elif isinstance(getattr(eldf, "params", None), dict):
        pdf_curve = np.asarray(eldf.params.get("pdf"), dtype=float).reshape(-1)

    if x_curve is None or cdf_curve is None or pdf_curve is None:
        raise RuntimeError("ELDF did not return expected curve arrays.")

    order = np.argsort(x_curve)
    x_curve = x_curve[order]
    cdf_curve = cdf_curve[order]
    pdf_curve = pdf_curve[order]

    ax2 = ax.twinx()

    l1, = ax.plot(x_curve, cdf_curve, color="blue", linewidth=1.8, label="ELDF")
    l2, = ax.step(y_sorted, empirical_cdf, where="post", color="green", linewidth=1.2, label="Empirical CDF")
    l3, = ax2.plot(x_curve, pdf_curve, color="#f0c987", linewidth=1.5, label="ELDF PDF")

    x_grid = np.linspace(y_sorted.min() - 2.0, y_sorted.max() + 2.0, 300)
    kde = gaussian_kde(y_sorted)
    l4, = ax2.plot(x_grid, kde(x_grid), color="red", linewidth=1.5, label="KDE PDF")

    ax.set_title(f"Data {ds_id} - y - varS={varS_flag}")
    ax.set_xlabel("Value")
    ax.set_ylabel("ELDF / CDF")
    ax2.set_ylabel("PDF")

    ax.grid(alpha=0.25)
    ax.set_ylim(0.04, 1.02)

    handles = [l1, l2, l3, l4]
    labels = [h.get_label() for h in handles]
    ax.legend(handles, labels, loc="upper left")

## Step 4: Create the final 2x4 presentation figure
This cell is the key output for your slide deck and is intentionally aligned with the reference style.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22, 10), constrained_layout=True)

for col, ds_id in enumerate([1, 2, 3, 4]):
    plot_eldf_panel(axes[0, col], datasets_y[ds_id], ds_id, varS_flag=False)

for col, ds_id in enumerate([1, 2, 3, 4]):
    plot_eldf_panel(axes[1, col], datasets_y[ds_id], ds_id, varS_flag=True)

plt.savefig("eldf_vs_cdf_pdf_y_2x4.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved: eldf_vs_cdf_pdf_y_2x4.png")

### Interpretation guide
- Dataset 1: smooth linear behavior
- Dataset 2: non-linear shape
- Dataset 3: outlier impact on tails
- Dataset 4: leverage structure with concentrated central mass